# Solutions · 08 · Controllability, observability and pole placement

Worked solutions to the exercises of [notebook 08](../notebooks/08_controllability_observability_and_pole_placement.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, statespace as sp, response as rs, models, linalg as la
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

For the cart-pendulum, place the poles at (-1 +- j, -2, -3), (-2 +- 2j, -5, -6) and (-4 +- 4j, -10, -12); simulate the recovery from 10 degrees and tabulate the settling time, the peak force and the cart excursion. Where does the actuator (say 10 N) rule out the design?

In [2]:
cp = models.cart_pendulum()["system"]
A, B, C = cp.A, cp.B, cp.C
rows = []
for des in ([-1 + 1j, -1 - 1j, -2.0, -3.0], [-2 + 2j, -2 - 2j, -5.0, -6.0], [-4 + 4j, -4 - 4j, -10.0, -12.0]):
    K = sp.place(A, B, des)
    Acl = sy.ss(A - B @ K, B, C, np.zeros((2, 1)))
    t, y, x = rs.initial(Acl, [0, 0, np.radians(10), 0], 6.0, 6001)
    u = -(x @ K.T)[:, 0]
    settled = t[np.max(np.flatnonzero(np.abs(x[:, 2]) > 0.02 * np.radians(10))) + 1]
    rows.append((str(np.round(des, 1)), settled, np.max(np.abs(u)), 100 * np.max(np.abs(x[:, 0]))))
print(pd.DataFrame(rows, columns=["poles", "angle settled (s)", "peak force (N)", "cart excursion (cm)"]).to_string(index=False, float_format="%.2f"))

                                poles  angle settled (s)  peak force (N)  cart excursion (cm)
    [-1.+1.j -1.-1.j -2.+0.j -3.+0.j]               4.98            1.91                56.98
    [-2.+2.j -2.-2.j -5.+0.j -6.+0.j]               2.41            4.72                19.36
[ -4.+4.j  -4.-4.j -10.+0.j -12.+0.j]               1.22           19.80                13.39


Faster poles mean a faster recovery and a smaller cart excursion, bought with force: the fastest set asks
for about 20 N, twice a 10 N actuator's limit, and would saturate - after which the linear design means
nothing. The second set is the practical choice. Pole placement does not know about actuators; the
designer must check, which is why LQR's cost weights are the more natural language (notebook 10).

## Exercise 2

The quarter car (`models.quarter_car`) has two inputs; is it controllable from the actuator force alone? From the road alone? Use the PBH test to name any mode that cannot be moved.

In [3]:
qc = models.quarter_car()["system"]
for j, name in ((1, "actuator force alone"), (0, "road alone")):
    c = sp.controllability(qc.A, qc.B[:, [j]])
    pbh = sp.pbh_test(qc.A, qc.B[:, [j]])
    print(f"{name:<22}: rank {c['rank']} of 4, singular values {np.array2string(c['singular_values'], precision=2)}; uncontrollable modes {np.round(pbh['uncontrollable_modes'], 2)}")
print("both inputs together: rank", sp.controllability(qc.A, qc.B)["rank"])

actuator force alone  : rank 4 of 4, singular values [5.04e+03 6.89e+00 4.14e-03 6.80e-04]; uncontrollable modes []
road alone            : rank 4 of 4, singular values [7.09e+08 1.07e+06 8.52e+01 7.62e+00]; uncontrollable modes []
both inputs together: rank 4


The quarter car is controllable from the actuator force alone and from the road alone - every mode can be
moved by either - but the singular values of the controllability matrices show that the road reaches the
wheel-hop mode much more easily than the body mode and the actuator the reverse. Controllability is a yes/no
property; the singular values say how much control it will take.

## Exercise 3

**Implement it yourself:** place the poles of the 5-state random system by the Sylvester method yourself (choose G, solve A X - X Lambda = B G with `linalg.sylvester`, K = G X^-1) and verify the closed-loop eigenvalues; try two different G and compare the gains.

In [4]:
rng = np.random.default_rng(2)
A5, B5 = rng.normal(size=(5, 5)), rng.normal(size=(5, 2))
des5 = np.array([-1, -2, -3 + 2j, -3 - 2j, -5], complex)
Lam = sp._real_block_diag(des5)
for seed in (0, 1):
    G = np.random.default_rng(seed).normal(size=(2, 5))
    X = la.sylvester(A5, -Lam, B5 @ G)                              # A X - X Lam = B G
    K = G @ np.linalg.inv(X)
    print(f"G from seed {seed}: closed-loop poles {np.round(np.sort_complex(np.linalg.eigvals(A5 - B5 @ K)), 6)}, |K| = {np.linalg.norm(K):.2f}")

G from seed 0: closed-loop poles [-5.+0.j -3.-2.j -3.+2.j -2.+0.j -1.+0.j], |K| = 71.63
G from seed 1: closed-loop poles [-5.+0.j -3.-2.j -3.+2.j -2.+0.j -1.+0.j], |K| = 33.37


With two inputs the gain matrix is not unique: any parameter matrix G (with X invertible) gives a K that
places the poles exactly, and different choices give gains of different size. The freedom is what robust
pole placement spends on well-conditioned closed-loop eigenvectors; the library's choice is simply the
smallest-norm gain among a few random tries.